# Llama Ablation — No Explanation Field

**Ablation purpose:** Isolate the effect of the `explanation` field in the model output.
In this variant the system message asks for JSON with **only** the `sense_id` key — the model
is never asked to produce a rationale.  `NEW_SENSE` remains allowed so this ablation tests
*only* the missing-explanation factor.

All other pipeline components are identical to `Llama_sense.ipynb`:
same model (`llama4`), temperature, chunk selection, processing loop, and writers.

Outputs are written with the `Llama4_noexp` origin label so they never overwrite baseline files.

## Model Information

| Property | Value |
|----------|-------|
| **Model** | Meta Llama 4 Scout |
| **Ollama Tag** | `llama4:latest` |
| **Temperature** | 0.0 |
| **Integration** | langchain-ollama |
| **Prompt Variant** | `Llama4_noexp` — sense_id only, NEW_SENSE allowed |

In [16]:
from config import ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR
from data_loader import load_sense_repo_by_round
from process_senses import process_senses_with_chain, default_build_senses_block, parse_model_output
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter
import time

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 2

# Set model origin for traceability (ablation: no explanation)
ORIGIN_LLM = "Llama4_noexp"


In [ ]:
# Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round_number=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 0  # 0 = 1-500, 1 = 501-1000, 2 = 1001-1500, ...
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

# For ablation runs, use a small slice by default so the notebook finishes quickly.
# Set test = False to run the full chunk.
test = True
if test:
    tb, te = 0, 20
    sentences = sentences[tb:te]
    begin = chunk_begin + tb
    end = chunk_begin + te - 1
    print(f"Using subset offsets {tb}:{te} -> absolute sentence range {begin}-{end}")
else:
    print(f"Using full chunk sentence range {begin}-{end}")


Loaded sense repo for round 2: 13283 senses
Available chunks (index, begin, end, file): [(0, 1, 500, 'sr-elexis-WSD_0001_0500.tsv'), (1, 501, 1000, 'sr-elexis-WSD_0501_1000.tsv'), (2, 1001, 1500, 'sr-elexis-WSD_1001_1500.tsv'), (3, 1501, 2000, 'sr-elexis-WSD_1501_2000.tsv'), (4, 2001, 2024, 'sr-elexis-WSD_2001_2024.tsv')]
Using chunk #1: sr-elexis-WSD_0501_1000.tsv -> (501, 1000)
Using subset offsets 0:100 -> absolute sentence range 501-600


In [18]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Ablation: JSON output must contain ONLY sense_id (no explanation field).
# NEW_SENSE is still allowed to isolate the missing-explanation factor.
system_message = """
Odgovorite isključivo validnim JSON-om u sledećem formatu:

{{
  "sense_id": "<TAČAN ID iz liste ili 'NEW_SENSE'>"
}}

VAŽNO:
- Odgovor MORA sadržati SAMO ključ "sense_id" — nemojte dodavati "explanation" niti bilo koji drugi ključ
- sense_id MORA biti IDENTIČAN jednom od ponuđenih ID-jeva (npr. "ENG30-00551215-n") ili tačno "NEW_SENSE"
- NIKADA ne koristite brojeve poput "1", "2", "značenje 1" itd.
- Ne dodajete nikakav tekst van JSON strukture
"""

user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

llm = ChatOllama(
    model="llama4",
    temperature=0.0,
)
parser = StrOutputParser()
chain = prompt | llm | parser

# Confirm origin label matches single source of truth.
assert ORIGIN_LLM == "Llama4_noexp"


In [ ]:
# --- DEBUG: Preview raw model outputs for early items ---
# Set DEBUG_PREVIEW = False to inspect a few raw model responses before the full run.
DEBUG_PREVIEW = False

if DEBUG_PREVIEW:
    from notebook_utils import debug_preview_chain
    debug_preview_chain(sentences, senses_df, chain, default_build_senses_block)


In [25]:
# Annotate senses using the shared utility
start_time = time.time()
sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM,
    build_senses_block=default_build_senses_block,
    parse_json_response_clean=parse_model_output
)
end_time = time.time()
print(f"Processed sentences {begin} to {end} in {end_time - start_time:.2f} seconds.")


Processed 100/100 sentences.Processed sentences 501 to 600 in 9068.75 seconds.


In [26]:
# Save outputs for the selected round and chunk
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")


In [27]:
# Write Inception-compatible output (for annotation import)
incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")


## Log processing time and completion

In [28]:
with open(OUTPUT_DIR / f"llama_{ORIGIN_LLM}_round{ROUND}.log", "a", encoding="utf-8") as f:
    f.write(f"Chunk {chunk_idx} | processed sentences {begin} to {end}\n")
    f.write(f"{ORIGIN_LLM} took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")


Done.
